# Semana 7: Transformers para entender a los usuarios de un asistente virtual

En la Semana 6 una red neuronal decidía qué máquinas detener a partir de **números de sensores**. Hoy el insumo es **texto**: frases que los usuarios le dicen a un asistente virtual. Para entenderlas usamos la arquitectura que está detrás de BERT y de ChatGPT: el **Transformer**.

Todo el modelo está escrito a mano en `src/clases_po/transformer_intenciones.py` (sin cajas negras), para poder señalar en el código cada operación que muestra la animación.

```
texto → tokens → ids → embeddings + posición → [atención → residual + norma → feed-forward → residual + norma] × 2 → promedio → 60 puntajes → softmax → ¿ejecutar o pasar a un humano?
```

## 🎬 Cómo usar este notebook con la animación

La clase se da con la animación **«Transformers en escena»** (10 escenas, unos 11 minutos). Cada sección lleva una marca **🎬 Escena N — Momento M**: cuando la animación se pausa al final de un momento, ejecuta las celdas con esa marca y explica el código. Debajo de la animación está el **laboratorio del forward pass**, con los mismos pesos de este modelo.

| Escena | Momentos | Sección |
|---|---|---|
| **1 · Todo empieza con una pregunta** | Frases sin pregunta · La pregunta · Desarmarla · Los candados | 1 |
| **2 · De un número por un peso a ChatGPT** | Lo que ya sabes · La escalera (1–5) · La escalera (6–8) · El límite | 2 |
| **3 · Tokens** | No lee como tú · ¿Qué es un token? · El vocabulario · Cada pieza es un número · Tokens en el caso real | 3 |
| **4 · Embeddings** | El casillero · Como en KNN · Cómo se aprenden · Las direcciones · El mapa · Sesgo y usos | 4 |
| **5 · Posición y atención** | El orden importa · ¿Qué banco? · Q, K, V · Varias cabezas | 5 |
| **6 · El forward pass** | Entrada · Una neurona · Una capa · Puntajes y softmax · Mezclar y normalizar · Feed-forward · Bloque 2 y salida | 6 |
| **7 · Las capas** | Las capas · ¿Cuántas? · ¿Por qué apilar? · ¿Más es mejor? | 7 |
| **8 · ¿Qué tan bien entiende?** | Los datos · Tres modelos · La lección · Confusiones | 8 |
| **9 · Ruido y alucinación** | Siempre responde algo · El ruido · Generar token a token · Cómo se controla | 9 |
| **10 · Ejecutar o pasar a un humano** | El umbral · La respuesta · La app · Cierre | 10 |

---
# Preparar el entorno

Instala las librerías nuevas (solo la primera vez): **PyTorch** (el modelo), **tokenizers** (entrenar el tokenizador WordPiece) y Streamlit + Plotly (la app).

In [ ]:
%pip install -q torch tokenizers "streamlit>=1.50" "plotly>=5.24" scikit-learn

In [48]:
#python -m pip install --upgrade pip

In [49]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "clases_po"))   # módulos sueltos, como la app

from transformer_intenciones import (AsistenteIntenciones, GeneradorFrases, TokenizadorWordPiece, MiniTransformer,
                                     cargar_massive, catalogo_intenciones, codificacion_posicional, FUERA_DE_DOMINIO)
from frontend_transformer import FrontendTransformer

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
torch.set_num_threads(2)
print("Raíz del proyecto:", ROOT, "· PyTorch", torch.__version__)

Raíz del proyecto: D:\IUDigital\2026\q4\machine learning\repositorios\ML_2026Q4 · PyTorch 2.14.1+cpu


**El modelo de referencia.** La animación y el laboratorio usan un Transformer ya entrenado, guardado en `src/sincronicas/modelos/transformer_intenciones.pt` (pesos, vocabulario y umbral). Lo cargamos aquí para que las cifras del notebook sean exactamente las de la animación. En la sección 4 puedes volver a entrenarlo (`ENTRENAR = True`); las cifras cambian un poco, porque el tokenizador puede numerar distinto las piezas empatadas y los pesos arrancan en otro punto.

In [51]:
MODELOS = ROOT / "src" / "sincronicas" / "modelos"
RUTA_MODELO = MODELOS / "transformer_intenciones.pt"
if RUTA_MODELO.exists():
    asistente = AsistenteIntenciones.cargar(RUTA_MODELO)
else:   # primera vez: no hay modelo de referencia → se entrena (≈ 2 minutos en CPU) y se guarda
    print("No se encontró", RUTA_MODELO.name, "→ entrenando el modelo de referencia…")
    _datos, _ = cargar_massive(ROOT / "src" / "sincronicas" / "datos" / "massive_es")
    asistente = AsistenteIntenciones(d=16, cabezas=2, d_ff=32, bloques=2, epocas=40, semilla=7).preparar(_datos).entrenar()
    asistente.guardar(RUTA_MODELO)
    asistente.exportar_json(MODELOS / "transformer_intenciones.json")
print(f"Modelo de referencia: d = {asistente.modelo.config['d']}, {asistente.modelo.config['bloques']} bloques, "
      f"vocabulario de {len(asistente.tok.vocab):,} piezas, umbral {asistente.umbral:.2f}")

Modelo de referencia: d = 16, 2 bloques, vocabulario de 2,000 piezas, umbral 0.87


---
# 1 · Todo empieza con una pregunta

> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 1 · Frases sin pregunta**

**Datos:** MASSIVE (Amazon Science, licencia CC BY 4.0), versión en español publicada en Hugging Face como `SetFit/amazon_massive_intent_es-ES`. Son frases que los usuarios le dicen a un asistente virtual, cada una con su **intención** (60 en total). `cargar_massive()` las descarga de Hugging Face la primera vez y las guarda en `src/sincronicas/datos/massive_es/`.

In [52]:
RUTA_DATOS = ROOT / "src" / "sincronicas" / "datos" / "massive_es"
datos, origen = cargar_massive(RUTA_DATOS)
asistente.preparar(datos, asistente.tok)            # usa el vocabulario guardado con el modelo
print("Origen:", origen)
print({k: len(v) for k, v in datos.items()})
datos["entrenamiento"].sample(10, random_state=1)

Origen: archivos locales en massive_es/
{'entrenamiento': 11514, 'validación': 2033, 'prueba': 2974}


,texto,etiqueta,intencion,intencion_es
5255,recuérdame cuando sea la hora de comer,50,calendar_set,conjunto de calendario
11463,cuál es el número de teléfono de martí,17,email_querycontact,email consultacontacto
4577,olly ayúdame a recordar que llame a joana a la...,50,calendar_set,conjunto de calendario
8939,cuéntame todo sobre albert einstein,49,qa_factoid,qa factoide
7200,lee mi lista,59,lists_query,consulta de listas
2525,cuál es la hora en la mitad del este de los es...,0,datetime_query,consulta de fecha y hora
1173,habla tan alto como pueda,14,audio_volume_up,subir el volumen de audio
7996,olly que puedo hacer divertido aqui alrededor,6,recommendation_events,eventos de recomendación
9890,ayuda micrófono,26,qa_definition,definición qa
3735,por favor hazme saber que alarma tengo para la...,23,alarm_query,consulta de alarma


> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 2 · La pregunta**

Eres el **líder de producto de un asistente virtual**. Miles de frases llegan cada día. Sin una pregunta, los datos no dicen qué hacer.

> ## ¿Qué pide cada usuario, para ejecutar la acción correcta al primer intento, y cuándo conviene pasarlo a un humano?

> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 3 · Desarmar la pregunta**

| Parte de la pregunta | Decisión técnica que obliga | Sección |
|---|---|---|
| **¿Qué pide…?** | Clasificar cada frase en una de 60 intenciones | 6, 8 |
| **…cada usuario…** | Una predicción por frase, en el momento | 6 |
| **…al primer intento…** | Medir aciertos (exactitud, F1) en frases que el modelo nunca vio | 8 |
| **…pasarlo a un humano** | Un umbral de confianza elegido con costos en pesos colombianos | 10 |

> 🎬 **Escena 1 · Todo empieza con una pregunta** — **Momento 4 · Los candados**

| 🔒 Pregunta hija | Dónde se responde |
|---|---|
| ¿Cómo lee el modelo? | 3 · Tokens |
| ¿Cómo entiende? | 4 · Embeddings, 5 · Atención, 6 · Forward pass |
| ¿Qué tan bien acierta? | 8 |
| ¿Cuándo se inventa cosas? | 9 |
| ¿Cuándo pasar a un humano? | 10 |
| ¿Cuánto ahorramos? | 10 |

In [53]:
catalogo = catalogo_intenciones(datos)
print(f"{len(catalogo)} intenciones en {catalogo['dominio'].nunique()} dominios")
catalogo.groupby("dominio")["intencion"].apply(lambda s: ", ".join(s)).to_frame("intenciones")

60 intenciones en 18 dominios


,intenciones
dominio,
alarm,"alarm_query, alarm_set, alarm_remove"
audio,"audio_volume_up, audio_volume_other, audio_vol..."
calendar,"calendar_remove, calendar_query, calendar_set"
cooking,"cooking_recipe, cooking_query"
datetime,"datetime_query, datetime_convert"
email,"email_addcontact, email_querycontact, email_se..."
general,"general_greet, general_quirky, general_joke"
iot,"iot_hue_lightchange, iot_wemo_off, iot_hue_lig..."
lists,"lists_createoradd, lists_remove, lists_query"


---
# 2 · De un número por un peso a ChatGPT

> 🎬 **Escena 2 · De un número por un peso a ChatGPT** — **Momento 1 · Lo que ya sabes**

Semana 6: una **neurona** calcula `z = w·x + b` y la pasa por una activación; una **red** apila neuronas en capas y aprende sus pesos con **descenso del gradiente**. Así anticipamos 47 de 51 fallas.

> 🎬 **Escena 2 · De un número por un peso a ChatGPT** — **Momento 2 · La escalera (1–5)** · > 🎬 **Escena 2 · De un número por un peso a ChatGPT** — **Momento 3 · La escalera (6–8)**

| Peldaño | Qué es | Dónde está en el código |
|---|---|---|
| 1 | Un número × un peso | cada multiplicación de `nn.Linear` |
| 2 | Suma ponderada | `x @ W.T` |
| 3 | + sesgo y una curva = **neurona** | `+ b`, `F.relu`, `softmax` |
| 4 | Muchas lado a lado = **capa** | `nn.Linear(d, d)` |
| 5 | Capas apiladas = **red neuronal** | `W1 → ReLU → W2` (feed-forward) |
| 6 | Cada palabra puntúa a las demás = **atención** | `BloqueTransformer.atencion()` |
| 7 | Atención + esa red = **bloque Transformer** | `BloqueTransformer.forward()` |
| 8 | Decenas de bloques + predecir la siguiente palabra = **GPT** | `GeneradorFrases` (causal) |

La celda siguiente es el peldaño 1 a 3, a mano: un número por un peso.

In [56]:
x = np.array([1.8, 2.1, -0.5])          # tres entradas
w = np.array([1.2, 1.5, -0.8])          # tres pesos
b = -4.0
print("productos:", x * w, "→ suma ponderada:", (x * w).sum(), "→ z = suma + b:", (x * w).sum() + b)
print("neurona (sigmoide):", 1 / (1 + np.exp(-((x * w).sum() + b))))

productos: [2.16 3.15 0.4 ] → suma ponderada: 5.710000000000001 → z = suma + b: 1.7100000000000009
neurona (sigmoide): 0.8468362842349139


> 🎬 **Escena 2 · De un número por un peso a ChatGPT** — **Momento 4 · El límite**

La red de la Semana 6 recibía **8 números fijos**. Una frase tiene **longitud variable**, **orden** y **significado**. Las secciones 3 a 6 resuelven eso.

---
# 3 · Tokens: cómo lee el modelo

> 🎬 **Escena 3 · Tokens** — **Momento 1 · No lee como tú**

El modelo no recibe palabras: antes de nada, un **tokenizador** corta la frase en piezas. Usamos **WordPiece**, el mismo esquema de BERT y BETO: se entrena con nuestras frases y luego aplica «la pieza más larga primero». El prefijo `##` indica «continuación de la palabra anterior».

In [57]:
tok_nuevo = TokenizadorWordPiece.entrenar(list(datos["entrenamiento"]["texto"]), tam_vocab=2000)   # así se entrena
tok = asistente.tok                                                                                  # el del modelo de referencia
for f in ["pon algo de shakira", "despiértame a las cinco", "¿va a llover mañana?"]:
    print(f"{f!r:32} → {tok.piezas(f)}")
print("¿Las mismas piezas que el recién entrenado?", all(tok.piezas(f) == tok_nuevo.piezas(f) for f in datos["prueba"]["texto"][:500]))

'pon algo de shakira'            → ['pon', 'algo', 'de', 'sh', '##a', '##k', '##ir', '##a']
'despiértame a las cinco'        → ['despiértame', 'a', 'las', 'cinco']
'¿va a llover mañana?'           → ['¿', 'va', 'a', 'llover', 'mañana', '[UNK]']
¿Las mismas piezas que el recién entrenado? True


> 🎬 **Escena 3 · Tokens** — **Momento 2 · ¿Qué es un token?**

Un token puede ser una **palabra entera** (`despiértame`), un **trozo** (`comp ##u ##tado ##ra`), **unas pocas letras** (`sh ##a ##k ##ir ##a`) o un **signo** (`¿`). Las palabras comunes caben en un token; las raras se parten.

> 🎬 **Escena 3 · Tokens** — **Momento 3 · Cómo se arma el vocabulario**

El vocabulario se aprende de los textos: empieza con letras y agrega las piezas que más ayudan a cubrir el corpus hasta llenar el tamaño pedido.

In [8]:
palabras = ["despiértame", "computadora", "alarma", "shakira"]
filas = {}
for v in [100, 500, 2000]:
    t = tok if v == 2000 else TokenizadorWordPiece.entrenar(list(datos["entrenamiento"]["texto"]), tam_vocab=v)
    filas[f"vocabulario de {v}"] = {p: " ".join(t.piezas(p)) for p in palabras}
pd.DataFrame(filas).T

,despiértame,computadora,alarma,shakira
vocabulario de 100,de ##s ##p ##i ##é ##r ##ta ##m ##e,c ##o ##m ##p ##u ##ta ##d ##or ##a,a ##l ##ar ##m ##a,s ##h ##a ##k ##i ##r ##a
vocabulario de 500,des ##p ##i ##ér ##ta ##me,comp ##u ##ta ##do ##ra,alarma,s ##h ##a ##k ##ir ##a
vocabulario de 2000,despiértame,comp ##u ##tado ##ra,alarma,sh ##a ##k ##ir ##a


> 🎬 **Escena 3 · Tokens** — **Momento 4 · Cada pieza es un número**

Cada pieza se cambia por su **id**: la posición en el vocabulario. El id **no significa nada**: es la dirección del casillero (la fila de la matriz de embeddings).

In [58]:
f = "pon algo de shakira"
pd.DataFrame({"token": tok.piezas(f), "id": tok.codificar(f)}).T

,0,1,2,3,4,5,6,7
token,pon,algo,de,sh,##a,##k,##ir,##a
id,150,399,82,1732,57,59,228,57


> 🎬 **Escena 3 · Tokens** — **Momento 5 · Tokens en el caso real**

¿Por qué importa? Los modelos tienen un **límite de tokens**, los servicios de IA **cobran por token**, y las palabras raras (marcas, nombres, errores de ortografía) se parten o quedan como `[UNK]`.

In [10]:
tr = list(datos["entrenamiento"]["texto"])
n_tok = [len(tok.codificar(t)) for t in tr]
n_pal = [len(TokenizadorWordPiece.palabras(t)) for t in tr]
ocurrencias = [w for t in tr for w in TokenizadorWordPiece.palabras(t)]
print(f"Tokens por frase: {np.mean(n_tok):.1f} · palabras por frase: {np.mean(n_pal):.1f} · máximo: {max(n_tok)} tokens")
print(f"Palabras que se parten en varias piezas: {np.mean([len(tok.piezas(w)) > 1 for w in ocurrencias]):.1%} de las apariciones")
print(f"Tokens [UNK] en la prueba: {sum(p == '[UNK]' for t in datos['prueba']['texto'] for p in tok.piezas(t))}")

Tokens por frase: 9.2 · palabras por frase: 7.3 · máximo: 60 tokens
Palabras que se parten en varias piezas: 14.9% de las apariciones
Tokens [UNK] en la prueba: 1


---
# 4 · Embeddings: el significado

Los embeddings se **aprenden** durante el entrenamiento (el detalle está en la sección 8). Por defecto usamos el modelo de referencia; con `ENTRENAR = True` lo entrenas aquí con estos hiperparámetros:

| Hiperparámetro | Valor | Qué es |
|---|---|---|
| `d` | 16 | números por token (dimensión del embedding) |
| `cabezas` | 2 | atenciones en paralelo (8 números cada una) |
| `d_ff` | 32 | neuronas de la feed-forward |
| `bloques` | 2 | bloques Transformer apilados |
| `epocas` | 40 | pasadas por los datos (se guarda la mejor en validación) |

In [59]:
ENTRENAR = False   # ← True para entrenarlo desde cero aquí (≈ 2 minutos en CPU; las cifras cambian un poco)
if ENTRENAR:
    asistente = AsistenteIntenciones(d=16, cabezas=2, d_ff=32, bloques=2, epocas=40, semilla=7)
    asistente.preparar(datos, tok).entrenar()
    tok = asistente.tok
print(f"Exactitud en prueba: {asistente.evaluar('prueba')['exactitud']:.1%} · umbral elegido en validación: {asistente.umbral:.2f}")

Exactitud en prueba: 77.5% · umbral elegido en validación: 0.87


> 🎬 **Escena 4 · Embeddings** — **Momento 1 · El casillero**

El id abre su fila en la matriz de embeddings (2 000 × 16): una lista de 16 números aprendidos.

In [60]:
E = asistente.modelo.E.weight.detach().numpy()
print("Matriz de embeddings:", E.shape)
pd.DataFrame([E[tok.id["pon"]]], index=["pon"], columns=[f"d{i+1}" for i in range(16)]).round(2)

Matriz de embeddings: (2000, 16)


,d1,d2,d3,d4,d5,d6,d7,d8,d9,d10,d11,d12,d13,d14,d15,d16
pon,-2.47,0.43,-1.27,0.86,-0.4,-1.77,0.03,0.5,-0.05,0.72,-0.54,-1.43,1.41,-0.75,1.26,-0.26


> 🎬 **Escena 4 · Embeddings** — **Momento 2 · Como en KNN**

En la clase de KNN cada cliente era un punto y los parecidos quedaban cerca. Aquí cada token es un punto; «parecido» se mide con la **similitud coseno**.

In [61]:
def vecinos(palabra, k=5):
    x = E[tok.id[palabra]]
    sim = E @ x / (np.linalg.norm(E, axis=1) * np.linalg.norm(x) + 1e-9)
    return [(tok.vocab[j], round(float(sim[j]), 2)) for j in np.argsort(-sim)[1:k + 1]]

pd.DataFrame({p: vecinos(p) for p in ["luces", "música", "clima", "correo", "alarma"]})

,luces,música,clima,correo,alarma
0,"(brillo, 0.87)","(cancion, 0.8)","(lloviendo, 0.83)","(email, 0.86)","(alarmas, 0.9)"
1,"(luz, 0.83)","(pop, 0.79)","(mej, 0.83)","(electrónico, 0.8)","(##erta, 0.86)"
2,"(brillante, 0.78)","(canciones, 0.77)","(tiempo, 0.83)","(publ, 0.71)","(pan, 0.8)"
3,"(habitación, 0.77)","(escuch, 0.76)","(fuera, 0.8)","(respon, 0.7)","(tend, 0.78)"
4,"(colores, 0.75)","(canción, 0.75)","(temperatura, 0.8)","(frío, 0.69)","(lla, 0.76)"


> 🎬 **Escena 4 · Embeddings** — **Momento 3 · Cómo se aprenden**

Las palabras que aparecen en contextos parecidos terminan cerca. Nuestro modelo aprendió sus embeddings **prediciendo intenciones**; los modelos grandes los aprenden leyendo millones de textos.

> 🎬 **Escena 4 · Embeddings** — **Momento 4 · Las direcciones significan algo**

*rey − hombre + mujer ≈ reina*: con embeddings entrenados con millones de textos (word2vec), la flecha de «hombre» a «mujer» es casi la misma que de «rey» a «reina». El resultado cae **cerca** de «reina» (se busca el vecino más cercano), no exactamente encima. Con vectores de juguete en 2D:

In [64]:
v = {"hombre": np.array([1.0, 1.0]), "mujer": np.array([1.0, 3.0]), "rey": np.array([5.0, 1.2]), "reina": np.array([5.1, 3.1])}
r = v["rey"] - v["hombre"] + v["mujer"]
print("rey − hombre + mujer =", r, "→ vecino más cercano:", min(v, key=lambda k: np.linalg.norm(v[k] - r)))

rey − hombre + mujer = [5.  3.2] → vecino más cercano: reina


> 🎬 **Escena 4 · Embeddings** — **Momento 5 · El mapa del caso real**

Los embeddings tienen 16 dimensiones; para dibujarlos se proyectan a 2 con **PCA** (se pierde información: es un mapa aproximado). BETO usa 768 dimensiones.

In [65]:
import plotly.express as px
from sklearn.decomposition import PCA

grupos = {"luces": ["luces", "luz", "brillo", "lámpara"], "música": ["música", "canción", "pop", "artista"],
          "clima": ["clima", "tiempo", "temperatura", "soleado"], "correo": ["correo", "email", "electrónico", "bandeja"],
          "alarma": ["alarma", "alarmas", "despiértame", "recordatorio"]}
pal = [(g, w) for g, ws in grupos.items() for w in ws if w in tok.id]
xy = PCA(2, random_state=0).fit_transform(E[[tok.id[w] for _, w in pal]])
fig = px.scatter(x=xy[:, 0], y=xy[:, 1], color=[g for g, _ in pal], text=[w for _, w in pal],
                 title="Embeddings reales del modelo (PCA de 16 a 2 dimensiones)")
fig.update_traces(textposition="top center", marker_size=12); fig.update_layout(template="plotly_white", height=480)
fig

> 🎬 **Escena 4 · Embeddings** — **Momento 6 · Sesgo y usos**

- **Sesgo:** el mapa aprende de lo que lee. Si los textos tienen prejuicios, el mapa también (por ejemplo, asociar una profesión con un solo género).
- **Usos:** buscadores que entienden lo que quisiste decir, chatbots que responden con tus documentos, recomendaciones, preguntas repetidas en atención al cliente. **Buscar significados es buscar vecinos.**

---
# 5 · Posición y atención

> 🎬 **Escena 5 · Posición y atención** — **Momento 1 · El orden importa**

«El perro mordió al hombre» y «el hombre mordió al perro» tienen las mismas palabras. A cada embedding se le suma una **huella de posición** sinusoidal:

$$PE_{p,2i} = \sin\left(\frac{p}{10000^{2i/d}}\right) \qquad PE_{p,2i+1} = \cos\left(\frac{p}{10000^{2i/d}}\right)$$

In [67]:
PE = codificacion_posicional(8, 16).numpy()
px.imshow(PE, color_continuous_scale="RdBu", zmin=-1, zmax=1, labels=dict(x="dimensión", y="posición"),
          title="Huella de posición (8 posiciones × 16 dimensiones)", aspect="auto")

> 🎬 **Escena 5 · Posición y atención** — **Momento 2 · ¿Qué banco?**

«Me senté en el **banco** del parque» y «fui al **banco** a pagar»: el embedding de «banco» es el mismo; el significado lo da el contexto.

> 🎬 **Escena 5 · Posición y atención** — **Momento 3 · Pregunta, clave y valor**

$$\text{Atención}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

Cada token hace una **pregunta** (Q), muestra su **clave** (K) y entrega su **valor** (V). En el código: `BloqueTransformer.atencion()`.

> 🎬 **Escena 5 · Posición y atención** — **Momento 4 · Varias cabezas**

Dos cabezas de 8 números cada una, en paralelo. Cada fila del mapa suma 1.

In [17]:
front = FrontendTransformer(asistente)
e = asistente.explicar("pon algo de shakira")
display(front.figura_atencion(e["atencion"][0][0], e["piezas"], "Bloque 1 · cabeza 1"))
front.figura_atencion(e["atencion"][0][1], e["piezas"], "Bloque 1 · cabeza 2")

---
# 6 · El forward pass, paso a paso

Recalculamos a mano, con NumPy y los **pesos reales** del modelo, todo lo que hace `MiniTransformer.forward()`. Al final comprobamos que da exactamente lo mismo que PyTorch. Es lo mismo que muestra el laboratorio de la animación.

> 🎬 **Escena 6 · El forward pass** — **Momento 1 · Entrada**

In [18]:
frase = "pon algo de shakira"
piezas, ids = tok.piezas(frase), tok.codificar(frase)
W = {k: v.detach().numpy() for k, v in asistente.modelo.state_dict().items()}
X = W["E.weight"][ids] + codificacion_posicional(len(ids), 16).numpy()     # embedding + posición
print("X:", X.shape, "(tokens × números)")
pd.DataFrame(X, index=piezas).round(2)

X: (8, 16) (tokens × números)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
pon,-2.47,1.43,-1.27,1.86,-0.40,-0.77,0.03,1.50,-0.05,1.72,-0.54,-0.43,1.41,0.25,1.26,0.74
algo,-0.21,0.25,0.52,0.79,0.74,0.52,-1.05,0.68,-0.27,1.01,0.11,0.41,-0.04,0.48,-0.65,2.27
de,0.24,-0.21,0.38,-0.13,-0.20,0.43,-0.42,1.62,0.49,0.94,-0.59,0.43,-0.28,-0.43,-0.31,1.91
sh,-1.16,-1.45,0.46,1.07,0.14,1.12,0.59,1.25,-0.90,1.93,-0.54,-0.30,-0.48,0.46,0.40,0.08
##a,-1.45,-0.63,0.83,0.07,0.46,0.88,0.50,1.18,0.12,1.26,-0.31,1.00,-0.21,1.35,-0.34,1.67
##k,-1.46,0.19,1.01,-0.59,1.33,0.88,-0.01,1.49,-0.12,1.75,-0.39,1.05,0.59,0.95,-1.07,2.71
##ir,-0.34,1.07,0.41,-2.05,0.19,1.31,0.73,1.57,1.35,-0.42,-0.99,1.32,-1.49,1.31,1.04,0.80
##a,-0.04,0.78,0.67,-0.83,0.72,0.72,0.60,1.16,0.15,1.26,-0.30,1.00,-0.21,1.35,-0.34,1.67


> 🎬 **Escena 6 · El forward pass** — **Momento 2 · Una neurona**

La neurona 1 de $W_Q$ con el token «pon»: 16 veces **un número por un peso**, la suma y el sesgo.

In [19]:
x, w, b = X[0], W["bloques.0.Wq.weight"][0], W["bloques.0.Wq.bias"][0]
productos = x * w
display(pd.DataFrame({"x": x, "peso": w, "x × peso": productos}).round(3).T)
print(f"Σ = {productos.sum():.3f} · + b ({b:.3f}) · z = {productos.sum() + b:.3f}")

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
x,-2.468,1.430,-1.273,1.858,-0.403,-0.772,0.031,1.496,-0.054,1.717,-0.541,-0.433,1.407,0.254,1.262,0.735
peso,0.004,-0.042,0.092,-0.106,-0.238,-0.141,0.187,0.127,0.088,0.354,0.141,-0.004,-0.572,0.073,-0.275,0.174
x × peso,-0.010,-0.060,-0.118,-0.198,0.096,0.109,0.006,0.189,-0.005,0.607,-0.076,0.002,-0.805,0.019,-0.347,0.128


Σ = -0.463 · + b (0.186) · z = -0.277


> 🎬 **Escena 6 · El forward pass** — **Momento 3 · Una capa**

16 neuronas lado a lado = una capa = multiplicar por una matriz: $Q = X W_Q^\top + b$.

In [68]:
lin = lambda A, k: A @ W[k + ".weight"].T + W[k + ".bias"]
Q, K, V = lin(X, "bloques.0.Wq"), lin(X, "bloques.0.Wk"), lin(X, "bloques.0.Wv")
print("Q, K, V:", Q.shape, "· la neurona de arriba es Q[0, 0] =", round(Q[0, 0], 3))

Q, K, V: (8, 16) · la neurona de arriba es Q[0, 0] = -0.277


> 🎬 **Escena 6 · El forward pass** — **Momento 4 · Puntajes y softmax**

In [21]:
def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

h, dk = 2, 8
cabezas = []
for c in range(h):
    sl = slice(c * dk, (c + 1) * dk)
    S = Q[:, sl] @ K[:, sl].T / np.sqrt(dk)        # puntajes
    A = softmax(S)                                  # cada fila suma 1
    cabezas.append(A @ V[:, sl])                    # mezcla de valores
    if c == 0:
        display(pd.DataFrame(A, index=piezas, columns=piezas).round(2).assign(suma=A.sum(1).round(2)))

,pon,algo,de,sh,##a,##k,##ir,##a,suma
pon,0.74,0.02,0.03,0.17,0.02,0.01,0.00,0.01,1.0
algo,0.80,0.00,0.01,0.18,0.00,0.00,0.00,0.00,1.0
de,0.69,0.02,0.02,0.24,0.01,0.00,0.00,0.01,1.0
sh,0.75,0.01,0.03,0.17,0.01,0.01,0.00,0.01,1.0
##a,0.77,0.00,0.01,0.21,0.00,0.00,0.00,0.00,1.0
##k,0.88,0.00,0.00,0.12,0.00,0.00,0.00,0.00,1.0
##ir,0.30,0.14,0.06,0.19,0.07,0.07,0.04,0.13,1.0
##a,0.73,0.01,0.01,0.24,0.00,0.00,0.00,0.00,1.0


> 🎬 **Escena 6 · El forward pass** — **Momento 5 · Mezclar, sumar y normalizar**

$O = \text{concat}(\text{cabezas})\,W_O^\top + b$ · residual $R = X + O$ · normalización $\frac{R - \mu}{\sigma} \cdot \gamma + \beta$

In [22]:
def layer_norm(A, k):
    mu, var = A.mean(-1, keepdims=True), A.var(-1, keepdims=True)
    return (A - mu) / np.sqrt(var + 1e-5) * W[k + ".weight"] + W[k + ".bias"]

O = lin(np.concatenate(cabezas, axis=1), "bloques.0.Wo")
R1 = X + O
z = (R1 - R1.mean(-1, keepdims=True)) / R1.std(-1, keepdims=True)
print(f"«pon» antes de normalizar: media {R1[0].mean():.2f}, desviación {R1[0].std():.2f} → después: media {z[0].mean():.2f}, desviación {z[0].std():.2f}")
N1 = layer_norm(R1, "bloques.0.norma1")

«pon» antes de normalizar: media 0.21, desviación 1.64 → después: media 0.00, desviación 1.00


> 🎬 **Escena 6 · El forward pass** — **Momento 6 · La red de la Semana 6**

Feed-forward: $\text{ReLU}(N W_1^\top + b_1)\,W_2^\top + b_2$, con 32 neuronas.

In [23]:
H = np.maximum(0, lin(N1, "bloques.0.W1"))                 # ReLU: las negativas se apagan
print(f"Neuronas apagadas en «pon»: {(H[0] == 0).sum()} de 32 · en toda la frase: {(H == 0).sum()} de {H.size}")
salida_bloque1 = layer_norm(N1 + lin(H, "bloques.0.W2"), "bloques.0.norma2")

Neuronas apagadas en «pon»: 26 de 32 · en toda la frase: 245 de 256


> 🎬 **Escena 6 · El forward pass** — **Momento 7 · Bloque 2 y salida**

El bloque 2 hace lo mismo con otros pesos. Después: promedio de los tokens → 60 puntajes → softmax. Y la comprobación contra PyTorch.

In [24]:
def bloque(Xb, i):
    p = f"bloques.{i}."
    Q, K, V = (lin(Xb, p + n) for n in ("Wq", "Wk", "Wv"))
    mezcla = np.concatenate([softmax(Q[:, c*dk:(c+1)*dk] @ K[:, c*dk:(c+1)*dk].T / np.sqrt(dk)) @ V[:, c*dk:(c+1)*dk] for c in range(h)], 1)
    N = layer_norm(Xb + lin(mezcla, p + "Wo"), p + "norma1")
    return layer_norm(N + lin(np.maximum(0, lin(N, p + "W1")), p + "W2"), p + "norma2")

salida = bloque(bloque(X, 0), 1)
probs = softmax(lin(salida.mean(0, keepdims=True), "salida"))[0]
top = np.argsort(-probs)[:5]
display(pd.DataFrame({"intención": catalogo["intencion"].to_numpy()[top], "probabilidad": probs[top].round(4)}))
print("¿Igual que PyTorch?", np.allclose(probs, asistente.predecir_proba([frase])[0], atol=1e-5))

,intención,probabilidad
0,play_music,0.9640
1,play_radio,0.0351
2,news_query,0.0004
3,music_settings,0.0002
4,play_audiobook,0.0001


¿Igual que PyTorch? True


---
# 7 · Las capas: cuáles, cuántas y por qué

> 🎬 **Escena 7 · Las capas** — **Momento 1 · Las capas del modelo**

In [25]:
p = asistente.modelo.n_parametros()
p["%"] = (p["parámetros"] / p["parámetros"].iloc[-1] * 100).round(1)
p

,capa,parámetros,%
0,Embeddings (vocabulario × d),32000,85.4
1,"Bloque 1 · atención (W_Q, W_K, W_V, W_O)",1088,2.9
2,"Bloque 1 · feed-forward (W₁, W₂)",1072,2.9
3,Bloque 1 · normalizaciones,64,0.2
4,"Bloque 2 · atención (W_Q, W_K, W_V, W_O)",1088,2.9
5,"Bloque 2 · feed-forward (W₁, W₂)",1072,2.9
6,Bloque 2 · normalizaciones,64,0.2
7,Cabeza de salida,1020,2.7
8,Total,37468,100.0


> 🎬 **Escena 7 · Las capas** — **Momento 2 · ¿Cuántas?**

| Modelo | Bloques | Pesos |
|---|---|---|
| Nuestro mini Transformer | 2 | ≈ 37 mil |
| BERT-base / BETO | 12 | ≈ 110 millones |
| GPT-3 | 96 | 175 mil millones |

Mismas ecuaciones en todos: cambian el tamaño `d` y cuántos bloques se apilan.

> 🎬 **Escena 7 · Las capas** — **Momento 3 · ¿Por qué apilar?**

Al entrar, «pon» es idéntico en dos frases distintas. Cada bloque lo reescribe con el contexto:

In [26]:
@torch.no_grad()
def representaciones(texto):
    ids_t = torch.tensor([tok.codificar(texto)]); m = ids_t != 0
    xb = asistente.modelo.E(ids_t) + asistente.modelo.PE[:ids_t.shape[1]]
    out = [xb[0, 0].numpy()]
    for b in asistente.modelo.bloques:
        xb, _ = b(xb, m); out.append(xb[0, 0].numpy())
    return out

coseno = lambda u, v: float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
for nombre, u, v in zip(["entrada", "después del bloque 1", "después del bloque 2"],
                        representaciones("pon una alarma a las siete"), representaciones("pon algo de shakira")):
    print(f"Similitud de «pon» en las dos frases · {nombre:22}: {coseno(u, v):.2f}")

Similitud de «pon» en las dos frases · entrada               : 1.00
Similitud de «pon» en las dos frases · después del bloque 1  : 0.93
Similitud de «pon» en las dos frases · después del bloque 2  : 0.40


> 🎬 **Escena 7 · Las capas** — **Momento 4 · ¿Más capas es mejor?**

El experimento tarda varios minutos (por eso está apagado). Resultados de referencia con la semilla 7: 1 bloque → 78,2 % en prueba; 2 bloques → 77,8 %; 4 bloques → 77,2 %; d = 64 con 2 bloques (≈ 199 mil pesos) → 77,8 % (y 97,5 % en entrenamiento). Con 11 514 frases, más capas casi no ayudan: el modelo memoriza.

In [27]:
EXPERIMENTO_CAPAS = False   # ← True para repetirlo (≈ 8 minutos en CPU)
if EXPERIMENTO_CAPAS:
    filas = []
    for d, hh, ff, bq in [(16, 2, 32, 1), (16, 2, 32, 2), (16, 2, 32, 4), (64, 4, 128, 2)]:
        m = AsistenteIntenciones(d=d, cabezas=hh, d_ff=ff, bloques=bq).preparar(datos, tok).entrenar(verbose=False)
        filas.append({"d": d, "bloques": bq, "parámetros": int(m.modelo.n_parametros().iloc[-1]["parámetros"]),
                      "exactitud entrenamiento": m.evaluar("entrenamiento")["exactitud"], "exactitud prueba": m.evaluar("prueba")["exactitud"]})
    display(pd.DataFrame(filas).round(3))

---
# 8 · ¿Qué tan bien entiende?

> 🎬 **Escena 8 · ¿Qué tan bien entiende?** — **Momento 1 · Los datos**

Entrenamiento (11 514): aprende los pesos · validación (2 033): elige la mejor época y el umbral · prueba (2 974): evaluación final.

In [28]:
conteo = datos["entrenamiento"]["intencion"].value_counts()
print(f"Frases por intención: mínimo {conteo.min()} ({conteo.idxmin()}) · máximo {conteo.max()} ({conteo.idxmax()})")
fig = front.figura_aprendizaje()
fig if fig is not None else print("La curva de aprendizaje aparece cuando entrenas el modelo (ENTRENAR = True en la sección 4).")

Frases por intención: mínimo 4 (cooking_query) · máximo 810 (calendar_set)


> 🎬 **Escena 8 · ¿Qué tan bien entiende?** — **Momento 2 · Tres modelos**

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

Xtr, ytr = datos["entrenamiento"]["texto"], datos["entrenamiento"]["etiqueta"]
Xte, yte = datos["prueba"]["texto"], datos["prueba"]["etiqueta"]
filas = []
for nombre, vec, clf in [("TF-IDF + regresión logística", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=3000, C=10)),
                         ("Red de la Semana 6 (MLP sobre TF-IDF)", TfidfVectorizer(sublinear_tf=True), MLPClassifier(hidden_layer_sizes=(64,), max_iter=60, random_state=7, early_stopping=True))]:
    v = vec.fit(Xtr); clf.fit(v.transform(Xtr), ytr); pred = clf.predict(v.transform(Xte))
    filas.append({"modelo": nombre, "exactitud": accuracy_score(yte, pred), "F1 macro": f1_score(yte, pred, average="macro")})
ev = asistente.evaluar("prueba")
filas.append({"modelo": "Mini Transformer desde cero", "exactitud": ev["exactitud"], "F1 macro": ev["F1 macro"]})
filas.append({"modelo": "XLM-R base preentrenado (FitzGerald et al., 2022)", "exactitud": 0.869, "F1 macro": np.nan})
pd.DataFrame(filas).round(3)

,modelo,exactitud,F1 macro
0,TF-IDF + regresión logística,0.808,0.782
1,Red de la Semana 6 (MLP sobre TF-IDF),0.807,0.761
2,Mini Transformer desde cero,0.775,0.737
3,"XLM-R base preentrenado (FitzGerald et al., 2022)",0.869,NaN


> 🎬 **Escena 8 · ¿Qué tan bien entiende?** — **Momento 3 · La lección**

Un Transformer pequeño **entrenado desde cero** con 11 mil frases no le gana a contar palabras. Los Transformers brillan cuando se **preentrenan** con millones de textos y luego se **ajustan** (*fine-tuning*) a la tarea: XLM-R base logra 86,9 % en es-ES. La celda opcional siguiente usa un modelo preentrenado multilingüe para obtener embeddings de frases (descarga ≈ 470 MB la primera vez).

In [30]:
USAR_PREENTRENADO = False   # ← True para probarlo (requiere: pip install sentence-transformers)
if USAR_PREENTRENADO:
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    emb_tr, emb_te = st_model.encode(list(Xtr), batch_size=64), st_model.encode(list(Xte), batch_size=64)
    clf = LogisticRegression(max_iter=3000, C=10).fit(emb_tr, ytr)
    pred = clf.predict(emb_te)
    print(f"Preentrenado + regresión logística: exactitud {accuracy_score(yte, pred):.3f} · F1 {f1_score(yte, pred, average='macro'):.3f}")

> 🎬 **Escena 8 · ¿Qué tan bien entiende?** — **Momento 4 · Dónde se confunde**

In [31]:
from sklearn.metrics import confusion_matrix
pred = asistente.predecir_proba(list(Xte)).argmax(1)
cm = confusion_matrix(yte, pred); np.fill_diagonal(cm, 0)
pares = np.dstack(np.unravel_index(np.argsort(-cm.ravel())[:8], cm.shape))[0]
pd.DataFrame([{"real": catalogo.intencion[i], "el modelo dijo": catalogo.intencion[j], "veces": cm[i, j]} for i, j in pares])

,real,el modelo dijo,veces
0,calendar_query,qa_factoid,24
1,general_quirky,alarm_set,21
2,alarm_set,general_quirky,18
3,play_radio,email_query,10
4,general_quirky,email_query,9
5,general_quirky,calendar_query,9
6,qa_factoid,calendar_query,9
7,general_quirky,news_query,8


---
# 9 · Ruido y alucinación

> 🎬 **Escena 9 · Ruido y alucinación** — **Momento 1 · Siempre responde algo**

El clasificador **siempre** elige una de las 60 intenciones, aunque la frase no tenga nada que ver con el asistente. Cuando lo hace con confianza alta, es una **alucinación**.

In [32]:
asistente.fuera_de_dominio(FUERA_DE_DOMINIO + ["me senté en el banco del parque"])

,frase,intención elegida,confianza,decisión
0,transfiere doscientos mil pesos a la cuenta de...,general_quirky,0.348,Pasar a un humano
1,mi pedido llegó roto y quiero que me devuelvan...,takeaway_order,0.803,Pasar a un humano
2,¿me aprobaron el crédito de vivienda?,qa_factoid,0.952,Ejecutar (¡alucina!)
3,quiero cancelar mi tarjeta de crédito,play_music,0.333,Pasar a un humano
4,agenda una cita con el dermatólogo de la eps,calendar_query,0.727,Pasar a un humano
5,asdf qwerty zxcv,qa_definition,0.205,Pasar a un humano
6,el gato azul baila con la luna en el tejado,iot_hue_lightchange,0.879,Ejecutar (¡alucina!)
7,¿cuántos puntos tengo en mi programa de millas?,general_quirky,0.395,Pasar a un humano
8,me senté en el banco del parque,qa_factoid,0.932,Ejecutar (¡alucina!)


> 🎬 **Escena 9 · Ruido y alucinación** — **Momento 2 · De dónde sale el ruido**

1. **Datos ruidosos:** etiquetas raras en el propio dataset.
2. **Palabras que no vio:** `[UNK]` y nombres partidos en muchos trozos.
3. **Softmax sobreconfiado:** la probabilidad no es certeza.
4. **Azar al generar:** la temperatura.

In [33]:
raras = pd.concat(datos.values())
display(raras[raras["texto"].isin(["el rosa es todo lo que necesitamos", "y la oscuridad ha caído"])][["texto", "intencion"]])
p = asistente.predecir_proba(list(Xte)); errores = p.argmax(1) != yte.to_numpy()
print(f"Errores en prueba: {errores.sum()} · de ellos, con confianza ≥ 90 %: {(p.max(1)[errores] >= 0.9).mean():.0%}")

,texto,intencion
2,el rosa es todo lo que necesitamos,iot_hue_lightchange
3,y la oscuridad ha caído,iot_hue_lighton


Errores en prueba: 668 · de ellos, con confianza ≥ 90 %: 20%


> 🎬 **Escena 9 · Ruido y alucinación** — **Momento 3 · Generar: token a token**

Un **decodificador** (tipo GPT) usa los mismos bloques, pero cada token solo mira hacia atrás (máscara causal) y la salida es el **siguiente token**. Lo entrenamos con las mismas frases (o lo cargamos si ya está guardado).

In [34]:
RUTA_GEN = ROOT / "src" / "sincronicas" / "modelos" / "generador_frases.pt"
try:
    generador = GeneradorFrases.cargar(RUTA_GEN)
    print("Generador cargado de", RUTA_GEN.name)
except Exception:
    generador = GeneradorFrases(tok, d=32, cabezas=2, d_ff=64, bloques=2, epocas=25).entrenar(list(datos["entrenamiento"]["texto"]))
    generador.guardar(RUTA_GEN)
generador.siguiente("pon una alarma a las", 6)

época   1 · pérdida 5.760
época   5 · pérdida 4.257
época  10 · pérdida 3.905
época  15 · pérdida 3.740
época  20 · pérdida 3.631
época  25 · pérdida 3.559


,token,probabilidad
0,ocho,0.2486
1,seis,0.1643
2,siete,0.1245
3,cinco,0.1149
4,dos,0.0885
5,doce,0.0570


In [35]:
for t in [0.3, 1.0, 1.8]:
    print(f"temperatura {t}:", [generador.generar("pon", temperatura=t, semilla=s) for s in range(3)])

temperatura 0.3: ['pon la canción', 'pon el último podcast de la radio', 'pon una alarma']
temperatura 1.0: ['pon música princesidad', 'pon audiolibro por favor', 'pon una alarma y oficina a las ocho y cinco de la tarde']
temperatura 1.8: ['pon usa princesidad neie no no juan sobre ejercicio', 'pon audiolibro por sus anllas a filicerca menta lanrár', 'ponje correo hasta última reunión con carche y mí siciona cómo cocinar españa']


Elige el token más **plausible**, no el **verdadero**: así se arma, token a token, una frase fluida pero inventada.

> 🎬 **Escena 9 · Ruido y alucinación** — **Momento 4 · Cómo se controla**

| Control | En este proyecto |
|---|---|
| Umbral de confianza → humano | `elegir_umbral()` (sección 10) |
| Detectar frases fuera de dominio | `fuera_de_dominio()` como prueba |
| Responder con documentos propios (RAG) | próxima clase: LLMs |
| Verificar antes de ejecutar | confirmar acciones críticas con el usuario |

---
# 10 · Ejecutar o pasar a un humano

> 🎬 **Escena 10 · Ejecutar o pasar a un humano** — **Momento 1 · El umbral se elige con dinero**

Supuestos (COP, ajustables en la app): **acción equivocada = 20 000** (reclamo, reproceso, cliente molesto) · **pasar a un humano = 6 000**.

$$\text{costo} = \text{errores al ejecutar} \times 20\,000 + \text{frases escaladas} \times 6\,000$$

El umbral se elige en **validación**; la prueba solo mide.

In [36]:
front.figura_costos(asistente.costo_error, asistente.costo_humano)

> 🎬 **Escena 10 · Ejecutar o pasar a un humano** — **Momento 2 · La respuesta**

In [37]:
r = asistente.respuesta_negocio()
print(f"Umbral: {r['umbral']:.2f}")
print(f"Resueltas solas: {r['% automáticas']:.1%} · pasan a un humano: {r['% a humano']:.1%} · errores al ejecutar: {r['% errores']:.1%}")
print(f"Costo con el asistente: COP {r['costo']:,.0f} · todo con humanos: COP {r['costo todo humano']:,.0f} · "
      f"ahorro: {r['ahorro %']:.0%} · sin umbral: COP {r['costo sin umbral']:,.0f}")
front.figura_reparto(r)

Umbral: 0.87
Resueltas solas: 68.3% · pasan a un humano: 31.7% · errores al ejecutar: 5.4%
Costo con el asistente: COP 8,872,000 · todo con humanos: COP 17,844,000 · ahorro: 50% · sin umbral: COP 13,360,000


> 🎬 **Escena 10 · Ejecutar o pasar a un humano** — **Momento 3 · La app y el laboratorio**

La app `src/clases_po/app_sinc7.py` carga los modelos guardados. También se exportan los pesos en JSON (es lo que usa el laboratorio de la animación).

In [38]:
GUARDAR = ENTRENAR   # solo se sobrescribe el modelo de referencia si lo reentrenaste
if GUARDAR:
    asistente.guardar(MODELOS / "transformer_intenciones.pt")
    asistente.exportar_json(MODELOS / "transformer_intenciones.json")
print("Modelos en", MODELOS.relative_to(ROOT), ":", sorted(p.name for p in MODELOS.glob("*.pt")))

Modelos en src\sincronicas\modelos : ['generador_frases.pt', 'transformer_intenciones.pt']


In [39]:
import torch as _t
requisitos = f'''--extra-index-url https://download.pytorch.org/whl/cpu
torch=={_t.__version__.split('+')[0]}
tokenizers
streamlit>=1.50
pandas>=2.2
numpy>=1.26
plotly>=5.24
scikit-learn
'''
APP = ROOT / "src" / "clases_po" / "app_sinc7.py"
(APP.parent / "requirements_sinc7.txt").write_text(requisitos, encoding="utf-8")
print(requisitos)

--extra-index-url https://download.pytorch.org/whl/cpu
torch==2.14.1
tokenizers
streamlit>=1.50
pandas>=2.2
numpy>=1.26
plotly>=5.24
scikit-learn



In [40]:
import logging
from streamlit.testing.v1 import AppTest
logging.getLogger("streamlit").setLevel(logging.ERROR)
at = AppTest.from_file(str(APP), default_timeout=600).run()
assert not at.exception, at.exception
print("Pestañas:", [t.label for t in at.tabs])
print("KPI:", [(k.label, k.value) for k in at.metric[:5]])
at.text_input(key="frase").set_value("me senté en el banco del parque").run()
assert not at.exception, at.exception
print("Decisión:", [x.value for x in at.success] + [x.value for x in at.warning])

2026-11-20 17:02:41.994 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Pestañas: ['🎯 Respuesta de negocio', '🔬 Dentro del Transformer', '📋 Validar un lote', '⚠️ Alucinación', '✍️ Siguiente palabra', '📈 Desempeño']
KPI: [('Resueltas solas', '68%'), ('Pasan a un humano', '32%'), ('Errores al ejecutar', '5.4%'), ('Costo con el asistente', 'COP 8,87 M'), ('Ahorro', 'COP 8,97 M')]
Decisión: ['**Ejecutar** · confianza 93.2%']


In [41]:
import subprocess
ABRIR_EN_NAVEGADOR = False   # ← True para levantar la app en http://localhost:8503
servidor = None
if ABRIR_EN_NAVEGADOR:
    servidor = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APP), "--server.port", "8503"], cwd=ROOT)
    print("App en http://localhost:8503")

> 🎬 **Escena 10 · Ejecutar o pasar a un humano** — **Momento 4 · Cierre**

La celda final abre los seis candados de la sección 1 con las cifras de esta ejecución.

In [42]:
acc = asistente.evaluar("prueba")["exactitud"]
fd = asistente.fuera_de_dominio(FUERA_DE_DOMINIO + ["me senté en el banco del parque"])
pd.DataFrame([
    ("¿Cómo lee el modelo?", f"WordPiece de {len(tok.vocab):,} piezas · {np.mean(n_tok):.1f} tokens por frase"),
    ("¿Cómo entiende?", "embeddings de 16 números + atención de 2 cabezas, 2 bloques"),
    ("¿Qué tan bien acierta?", f"{acc:.1%} (mini, desde cero) · 86,9 % un Transformer preentrenado"),
    ("¿Cuándo se inventa cosas?", f"{(fd['decisión'].str.startswith('Ejecutar')).sum()} de {len(fd)} frases fuera de dominio se ejecutarían"),
    ("¿Cuándo pasar a un humano?", f"cuando la confianza es menor que {asistente.umbral:.2f}"),
    ("¿Cuánto ahorramos?", f"{r['ahorro %']:.0%} frente a atender todo con humanos"),
], columns=["🔓 pregunta", "respuesta"])

,🔓 pregunta,respuesta
0,¿Cómo lee el modelo?,"WordPiece de 2,000 piezas · 9.2 tokens por frase"
1,¿Cómo entiende?,embeddings de 16 números + atención de 2 cabez...
2,¿Qué tan bien acierta?,"77.5% (mini, desde cero) · 86,9 % un Transform..."
3,¿Cuándo se inventa cosas?,3 de 9 frases fuera de dominio se ejecutarían
4,¿Cuándo pasar a un humano?,cuando la confianza es menor que 0.87
5,¿Cuánto ahorramos?,50% frente a atender todo con humanos


**El texto se vuelve tokens, los tokens números, los números significados, y los significados decisiones.** Y todo empezó con un número por un peso.

## Ejercicios propuestos

1. Cambia `tam_vocab` a 500 y a 8 000. ¿Qué pasa con los tokens por frase y con la exactitud?
2. Entrena con `d=32` y 4 cabezas. ¿Mejora la prueba o solo el entrenamiento?
3. Pon `USAR_PREENTRENADO = True` y compara con el mini Transformer. ¿Cuánto cambia la respuesta de negocio?
4. Propón una regla para detectar frases fuera de dominio (por ejemplo, la distancia del embedding promedio a los ejemplos de entrenamiento) y mide cuántas alucinaciones evita.